# 02 — Data Preprocessing

Phase 2: clean `dataset/raw/train.csv` (read-only), fit a sklearn `Pipeline` on the **training** split only, write `dataset/processed/` and `models/preprocessor.pkl`.

Logic lives in `ml/preprocess.py`. This notebook inspects before/after stats and output shapes. **No model training.**

In [1]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "ml" / "preprocess.py").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ml.preprocess import (
    CLIP_NEGATIVES_TO_ZERO,
    REQUIRED_FEATURES,
    run_preprocessing,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)
print("ROOT:", ROOT)

ROOT: G:\SLIIT Academy\6th Semester\FDM\FDM Project\FDM_Project


## 1. Raw file (before any transform)

`dataset/raw/train.csv` is not written to.

In [2]:
raw = pd.read_csv(ROOT / "dataset" / "raw" / "train.csv")
print("raw shape:", raw.shape)
print("missing cells:", int(raw.isna().sum().sum()))

inspect_cols = CLIP_NEGATIVES_TO_ZERO + ["product_rating", "delivery_delay_days"]
before_stats = raw[inspect_cols].agg(["min", "max", "mean", "median", "std"]).T
before_stats["n_negative"] = [(raw[c] < 0).sum() for c in inspect_cols]
before_stats["n_lt_1"] = [(raw[c] < 1).sum() if c == "product_rating" else np.nan for c in inspect_cols]
before_stats["n_gt_5"] = [(raw[c] > 5).sum() if c == "product_rating" else np.nan for c in inspect_cols]
before_stats["n_gt_100"] = [(raw[c] > 100).sum() if c == "discount_percent" else np.nan for c in inspect_cols]
before_stats["n_gt_1"] = [(raw[c] > 1).sum() if c == "past_return_rate" else np.nan for c in inspect_cols]
print("\nBefore cleaning:")
display(before_stats.round(4))

raw shape: (200000, 16)
missing cells: 0

Before cleaning:


,min,max,mean,median,std,n_negative,n_lt_1,n_gt_5,n_gt_100,n_gt_1
product_price,-183.2921,855.0786,37.9842,24.4611,45.6269,27285,NaN,NaN,NaN,NaN
discount_percent,-7.5408,78.7915,42.8721,42.9452,21.5105,1686,NaN,NaN,0.0,NaN
past_return_rate,-0.0609,0.8664,0.2151,0.1914,0.1372,2220,NaN,NaN,NaN,0.0
session_length_minutes,-11.0105,137.6529,77.4061,81.5623,36.3942,1487,NaN,NaN,NaN,NaN
num_product_views,-5.0000,44.0000,14.6544,13.0000,12.6594,21443,NaN,NaN,NaN,NaN
product_rating,0.4453,5.4527,2.8128,2.8187,1.2199,0,14280.0,3831.0,NaN,NaN
delivery_delay_days,-9.0500,7.9613,0.6251,0.5190,2.3312,78564,NaN,NaN,NaN,NaN


## 2. Run `ml.preprocess`

Deterministic clips → stratified 80/20 split → fit `FeatureCleaner` (train p99 price cap) + `ColumnTransformer` (StandardScaler + OneHotEncoder) on **train only**.

In [3]:
result = run_preprocessing()
log = result["log"]
print("\nproduct_price cap (train p99):", result["price_cap"])
print("rows dropped:", log["rows_dropped"])
print(log["decision_product_price_tail"])

Phase 2 complete.
  product_price cap (train p99): 188.582191
  X_train (160000, 26)  X_test (40000, 26)
  y_train (160000,)  y_test (40000,)
  preprocessor -> G:\SLIIT Academy\6th Semester\FDM\FDM Project\FDM_Project\models\preprocessor.pkl
  cleaned splits -> G:\SLIIT Academy\6th Semester\FDM\FDM Project\FDM_Project\dataset\processed

product_price cap (train p99): 188.5821914774632
rows dropped: 0
Winsorize/cap product_price at the training 99th percentile instead of dropping high-price rows. Cap is stored on FeatureCleaner.price_cap_ inside preprocessor.pkl so inference uses the same threshold.


## 3. Before / after for every clipped or capped column

After stats come from the cleaned train+test tables (same 200,000 rows; none dropped). `delivery_delay_days` is shown as a control — it must still have negatives.

In [4]:
train_c = pd.read_csv(ROOT / "dataset" / "processed" / "train.csv")
test_c = pd.read_csv(ROOT / "dataset" / "processed" / "test.csv")
cleaned = pd.concat([train_c, test_c], ignore_index=True)

assert len(cleaned) == len(raw)
assert set(REQUIRED_FEATURES + ["returned"]) == set(cleaned.columns)
assert "order_id" not in cleaned.columns

rows = []
for col in inspect_cols:
    b, a = raw[col], cleaned[col]
    rows.append(
        {
            "column": col,
            "min_before": b.min(),
            "min_after": a.min(),
            "max_before": b.max(),
            "max_after": a.max(),
            "mean_before": b.mean(),
            "mean_after": a.mean(),
            "n_negative_before": int((b < 0).sum()),
            "n_negative_after": int((a < 0).sum()),
        }
    )
cmp = pd.DataFrame(rows)
display(cmp.round(4))

print("\nClip impact from preprocessing_log.json:")
display(pd.DataFrame(log["clip_impact"]).T)

print("\nproduct_rating after: min/max =", cleaned["product_rating"].min(), cleaned["product_rating"].max())
print("discount_percent after: min/max =", cleaned["discount_percent"].min(), cleaned["discount_percent"].max())
print("past_return_rate after: min/max =", cleaned["past_return_rate"].min(), cleaned["past_return_rate"].max())
print("product_price after: min/max =", cleaned["product_price"].min(), cleaned["product_price"].max())
print("delivery_delay_days still signed: n_negative =", int((cleaned["delivery_delay_days"] < 0).sum()))

,column,min_before,min_after,max_before,max_after,mean_before,mean_after,n_negative_before,n_negative_after
0,product_price,-183.2921,0.00,855.0786,188.5822,37.9842,37.9912,27285,0
1,discount_percent,-7.5408,0.00,78.7915,78.7915,42.8721,42.8902,1686,0
2,past_return_rate,-0.0609,0.00,0.8664,0.8664,0.2151,0.2152,2220,0
3,session_length_minutes,-11.0105,0.00,137.6529,137.6529,77.4061,77.4317,1487,0
4,num_product_views,-5.0000,0.00,44.0000,44.0000,14.6544,14.8661,21443,0
5,product_rating,0.4453,1.00,5.4527,5.0000,2.8128,2.8241,0,0
6,delivery_delay_days,-9.0500,-9.05,7.9613,7.9613,0.6251,0.6251,78564,78564



Clip impact from preprocessing_log.json:


,n_changed,pct_changed,min_before,max_before,min_after,max_after,n_negative_before,n_negative_after
product_price,29324.0,14.6620,-183.292092,855.078645,0.0,188.582191,27285.0,0.0
discount_percent,1686.0,0.8430,-7.540832,78.791547,0.0,78.791547,1686.0,0.0
past_return_rate,2220.0,1.1100,-0.060878,0.866420,0.0,0.866420,2220.0,0.0
session_length_minutes,1487.0,0.7435,-11.010479,137.652911,0.0,137.652911,1487.0,0.0
num_product_views,21443.0,10.7215,-5.000000,44.000000,0.0,44.000000,21443.0,0.0
product_rating,18111.0,9.0555,0.445342,5.452690,1.0,5.000000,0.0,0.0



product_rating after: min/max = 1.0 5.0
discount_percent after: min/max = 0.0 78.79154678672955
past_return_rate after: min/max = 0.0 0.8664200331772883
product_price after: min/max = 0.0 188.5821914774632
delivery_delay_days still signed: n_negative = 78564


## 4. Transformed matrices and preprocessor artifact

In [5]:
X_train = np.load(ROOT / "dataset" / "processed" / "X_train.npy")
X_test = np.load(ROOT / "dataset" / "processed" / "X_test.npy")
y_train = np.load(ROOT / "dataset" / "processed" / "y_train.npy")
y_test = np.load(ROOT / "dataset" / "processed" / "y_test.npy")
feature_names = json.loads((ROOT / "dataset" / "processed" / "feature_names.json").read_text())

print("X_train", X_train.shape)
print("X_test ", X_test.shape)
print("y_train", y_train.shape, "positive rate", y_train.mean().round(4))
print("y_test ", y_test.shape, "positive rate", y_test.mean().round(4))
print("n features", len(feature_names))
print("feature names:")
print(feature_names)

pre = joblib.load(ROOT / "models" / "preprocessor.pkl")
print("\nloaded", type(pre).__name__, "steps:", list(pre.named_steps))
print("stored price_cap_", pre.named_steps["clean"].price_cap_)

roundtrip = pre.transform(raw[REQUIRED_FEATURES].head(3))
print("inference smoke test (3 rows):", roundtrip.shape)

X_train (160000, 26)
X_test  (40000, 26)
y_train (160000,) positive rate 0.4746
y_test  (40000,) positive rate 0.4746
n features 26
feature names:
['num__customer_age', 'num__product_price', 'num__discount_percent', 'num__product_rating', 'num__past_purchase_count', 'num__past_return_rate', 'num__delivery_delay_days', 'num__session_length_minutes', 'num__num_product_views', 'cat__device_type_mobile', 'cat__device_type_desktop', 'cat__device_type_tablet', 'cat__product_category_toys', 'cat__product_category_beauty', 'cat__product_category_electronics', 'cat__product_category_home', 'cat__product_category_clothing', 'cat__product_category_sports', 'cat__shipping_method_standard', 'cat__shipping_method_express', 'cat__shipping_method_same_day', 'cat__payment_method_debit_card', 'cat__payment_method_credit_card', 'cat__payment_method_apple_pay', 'cat__payment_method_paypal', 'bin__used_coupon']

loaded Pipeline steps: ['clean', 'encode_scale']
stored price_cap_ 188.5821914774632
inference 

## 5. Checks that must hold

- No rows dropped (200,000 in, 160,000 + 40,000 out).
- Invalid-sign columns have **zero** negatives after cleaning, except `delivery_delay_days`.
- `product_rating` in [1, 5], `discount_percent` in [0, 100], `past_return_rate` in [0, 1].
- `product_price` max equals the **training** 99th percentile cap (logged; not hardcoded).
- `order_id` absent; `returned` only as `y_*` / CSV target.
- Preprocessor fitted on train only; pickle is what Flask will load later.